# Fencing bout segmenter

Thin driver for the `fenceseg` package. All logic lives in the package so it
can be version-controlled, tested and run headless; this notebook just wires it
up for Kaggle / Colab.

**The start and end conditions are unchanged from the original notebook.**
`tools/verify_state_machine.py` fuzzes the refactored state machine against a
verbatim transcription of the original over ~26,000 sequences and requires an
exact match.

## 1. Setup

In [ ]:
!pip -q install yt-dlp pytesseract opencv-python-headless
!apt-get -qq update && apt-get -qq install -y ffmpeg tesseract-ocr

# yolov5 runtime deps (torch.hub pulls the repo itself on first Detector build)
!pip -q install "torch>=2.0" torchvision seaborn psutil

In [ ]:
import sys, pathlib
REPO = pathlib.Path('/kaggle/working/fencing-segmenter')   # adjust if needed
sys.path.insert(0, str(REPO))

from fenceseg.config import Config
from fenceseg.pipeline import analyse, build_bouts, write_report, process
from fenceseg.cut import cut_all

## 2. Verify the conditions are untouched

Run this once. It should print `PASS`. If you ever change `fenceseg/segment.py`,
run it again.

In [ ]:
!cd /kaggle/working/fencing-segmenter && python tools/verify_state_machine.py

## 3. Get the video

FencingTV serves HLS. Get the URL from Chrome DevTools:
F12 -> Network -> filter `.m3u8` -> click the entry with the random-looking
name (**not** the ones labelled `rendition`) -> copy the Request URL.

`concurrent_fragments` is the main download speedup: HLS is thousands of small
files and yt-dlp fetches them serially by default.

In [ ]:
from fenceseg.download import download

URL = "PASTE_THE_M3U8_REQUEST_URL_HERE"
video = download(URL, pathlib.Path('/kaggle/working/work/stream_01.mp4'),
                 concurrent_fragments=8)
print(video)

In [ ]:
# Already have a local file? Skip the cell above and just point at it.
# video = pathlib.Path('/kaggle/input/my-stream/stream.mp4')

## 4. Configure

In [ ]:
cfg = Config(
    weights = pathlib.Path('/kaggle/input/newfencingmodel/best.pt'),
    workdir = pathlib.Path('/kaggle/working/work'),
    outdir  = pathlib.Path('/kaggle/working/bouts'),

    sample_fps = 1.0,        # decision rate; 1.0 == the original notebook
    batch_size = 32,         # throughput only, does not affect decisions
    hwaccel    = 'cuda',     # set to None if ffmpeg has no CUDA decoder
    device     = 'cuda:0',
    half       = True,

    use_templates = True,    # learned digit templates: the big OCR speedup
    temporal_vote = False,   # see README before enabling

    cut_mode = 'copy',       # 'reencode' for frame-accurate cuts
)
cfg

## 5. Analyse, then cut

One pass over the video collects scores, fencer counts and name plates;
boundaries and filenames are resolved afterwards.

In [ ]:
records, boundaries, stats = analyse(video, cfg)
bouts = build_bouts(records, boundaries, cfg)
write_report(video, records, boundaries, bouts, stats, cfg)

print(f"\n{len(bouts)} bouts\n")
for b in bouts:
    print(f"{b.start:8.1f} -> {b.end:8.1f}   {b.filename}")

### Check the names before cutting

If a filename looks wrong, the raw plate text is on the bout object. Fixing a
name here is much cheaper than re-cutting.

In [ ]:
for b in bouts:
    print(f"{b.index:3d}  L={b.left_plate!r}  R={b.right_plate!r}")

# Manual override example:
# bouts[3].filename = "Alexander Massialas USA vs. Race Imboden USA"

In [ ]:
jobs = [(b.start, b.end, b.filename) for b in bouts]
written = cut_all(video, jobs, cfg.outdir, cfg.cut_mode)
print(f"\n{len(written)} files written to {cfg.outdir}")

## 6. Optional: harvest frames to improve the detector

Writes the frames the model is least sure about, with pre-filled labels.
See `training/README.md`.

In [ ]:
!cd /kaggle/working/fencing-segmenter && python tools/harvest_frames.py \
    {video} --weights {cfg.weights} --out /kaggle/working/dataset/candidates \
    --per-bucket 150
